<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/BI_Scraping_MEF_DEV_Funcion_GobiernoRegional_LOR_SMN_UCA_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests beautifulsoup4 pandas numpy tqdm google-api-python-client --quiet
print("✅ Librerías listas")
import os, re, time, random, zipfile
import requests
import numpy as np
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

✅ Librerías listas


In [2]:
# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
AÑO = 2025
MES_INICIO = 1                 # primer mes del bucle
MES_FIN = None                 # None = último mes disponible
INCLUIR_MES_EN_CURSO = False   # False = solo meses cerrados (regla del scraper anterior)

CONTROL_ANUAL = True           # compara contra la vista anual de Consulta Amigable
INCLUIR_ENTIDAD = True         # añade columna ENTIDAD = "GOBIERNO REGIONAL DE <REGIÓN>"
SUBIR_A_DRIVE = True           # sube cada CSV a la carpeta de Drive de arriba
DRIVE_FOLDER_ID = "18Ohcjmg6FojhNwbCHUHX3MEOxhokZA9b"
DESCARGA_FINAL = "ninguna"     # "zip" | "individual" | "ninguna"

# ── Filtros de la ruta (los mismos del scraper municipal, cambiando el nivel) ──
NIVEL_GOBIERNO = "R"                          # R = Gobiernos Regionales (M era Locales)
NIVEL_ETIQUETA = "R: GOBIERNOS REGIONALES"
GENERICA = "6-2-6"                            # Genérica 6-26 (como en tu captura)
PARAMS_EXTRA = {}   # si F12>Network muestra otros parámetros, se sobreescriben aquí

PAUSA = (0.4, 1.0)             # segundos aleatorios entre consultas (cortesía con el servidor)

# ── Meses válidos ────────────────────────────────────────────────────────────
hoy = date.today()
if AÑO > hoy.year:
    raise ValueError("Año inválido: no puede ser futuro.")
if AÑO < hoy.year:
    meses_validos = list(range(1, 13))
else:
    tope = hoy.month if INCLUIR_MES_EN_CURSO else hoy.month - 1
    if tope < 1:
        raise ValueError("No hay meses cerrados aún en este año.")
    meses_validos = list(range(1, tope + 1))
ultimo_valido = meses_validos[-1]
if MES_FIN is None:
    MES_FIN = ultimo_valido
if not (1 <= MES_INICIO <= MES_FIN <= ultimo_valido):
    raise ValueError(f"Rango inválido ({MES_INICIO}-{MES_FIN}). Disponibles: 1 a {ultimo_valido}.")
meses = [str(m) for m in range(MES_INICIO, MES_FIN + 1)]

# ── Catálogos ────────────────────────────────────────────────────────────────
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTÍN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]
nombres_meses = ["Enero","Febrero","Marzo","Abril","Mayo","Junio","Julio","Agosto",
                 "Setiembre","Octubre","Noviembre","Diciembre"]
meses_map  = {str(i): f"{i}: {nombres_meses[i-1]}" for i in range(1, 13)}
deptos_map = {d["codigo"]: d["departamento"] for d in departamentos}

ruta_guardado = "/content/Inversion_Funcion.GobiernoRegional"
os.makedirs(ruta_guardado, exist_ok=True)
SUFIJO_REGIONES = "LOR - SMN - UCA"

def etiqueta_meses(ms):
    nums = sorted(int(m) for m in ms)
    if len(nums) == 1:
        return f"Mes {nums[0]:02d}"
    if nums == list(range(nums[0], nums[-1] + 1)):
        return f"Meses {nums[0]:02d}-{nums[-1]:02d}"
    return "Meses " + ",".join(f"{m:02d}" for m in nums)

def nombre_archivo_salida(año, ms):
    return f"DEV Funcion.GobRegional {año} - {etiqueta_meses(ms)} - {SUFIJO_REGIONES}.csv"

COLUMNAS_SALIDA = (["AÑO", "MES", "NIVEL_GOBIERNO", "DEPARTAMENTO"]
                   + (["ENTIDAD"] if INCLUIR_ENTIDAD else [])
                   + ["FUNCION", "DEVENGADO"])
print(f"Año {AÑO} -> meses a procesar: {', '.join(meses)}  |  regiones: "
      f"{', '.join(d['departamento'] for d in departamentos)}")

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
drive_subidos, drive_fallidos = [], []
STATUS_REINTENTABLES = {403, 429, 500, 502, 503, 504}

def conectar_drive(folder_id):
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    try:
        carpeta = servicio.files().get(
            fileId=folder_id, fields="id,name,mimeType,capabilities(canAddChildren)",
            supportsAllDrives=True).execute()
    except HttpError as e:
        raise PermissionError(f"Sin acceso a la carpeta de Drive (HTTP {e.resp.status}).") from e
    if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError("El ID no corresponde a una carpeta de Drive.")
    if not carpeta.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
    print(f"☁️  Drive conectado -> carpeta: '{carpeta['name']}'\n")
    return servicio

def subir_a_drive(servicio, ruta_local, folder_id, max_reintentos=5):
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    nombre = os.path.basename(ruta_local)
    mime = "application/zip" if nombre.lower().endswith(".zip") else "text/csv"
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    ultimo_error = ""
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(
                q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                fields="files(id)", supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media,
                                        fields="id", supportsAllDrives=True).execute()
                accion = "actualizado"
            else:
                servicio.files().create(body={"name": nombre, "parents": [folder_id]},
                                        media_body=media, fields="id",
                                        supportsAllDrives=True).execute()
                accion = "creado"
            drive_subidos.append(nombre)
            print(f"   ☁️ Drive ({accion}): {nombre}")
            return True
        except HttpError as e:
            ultimo_error = f"HTTP {e.resp.status}"
            if e.resp.status not in STATUS_REINTENTABLES:
                break
        except Exception as e:
            ultimo_error = str(e)
        if intento < max_reintentos:
            espera = 2 ** intento + random.random()
            print(f"   ↻ Reintento {intento}/{max_reintentos - 1} en {espera:.1f}s ({ultimo_error})")
            time.sleep(espera)
    drive_fallidos.append((nombre, ultimo_error))
    print(f"   ❌ No se pudo subir {nombre} ({ultimo_error}). Sigue en {ruta_local}")
    return False

drive = conectar_drive(DRIVE_FOLDER_ID) if SUBIR_A_DRIVE else None

# ── 3. EXTRACCIÓN: UNA CONSULTA POR REGIÓN Y MES DEVUELVE TODAS LAS FUNCIONES ─
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"})

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|"
                     r"funci[oó]n|mes\b|trimestre|sector|pliego|fuente|rubro|categor|"
                     r"producto|divisi[oó]n)", re.I)
_DETALLE = re.compile(r"^\d{2}\s*:")

def parsear(html):
    """Devuelve dict(cabecera, filtros, filas) o None si la página no trae tabla.
    filtros: filas del 'breadcrumb' (TOTAL, Genérica, Nivel de Gobierno, Departamento…)
    filas  : (código, nombre, devengado) de cada función.
    El Devengado es la 3.ª celda desde el final (… Devengado | Girado | Avance %)."""
    pagina = BeautifulSoup(html, "html.parser")
    if pagina.find("table", class_="Data") is None:
        return None
    cabecera, filtros, filas = "", [], []
    for tr in pagina.find_all("tr"):
        if tr.find("table"):
            continue
        if tr.find("th"):
            if not cabecera:
                cabecera = tr.find("th").get_text(" ", strip=True)
            continue
        celdas = tr.find_all("td")
        if len(celdas) < 8:
            continue
        textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:2]]
        etiqueta = next((t for t in textos if t), "")
        dev = a_numero(celdas[-3].get_text(strip=True))
        if dev != dev:                                   # NaN -> no es fila numérica
            continue
        if _DETALLE.match(etiqueta):
            filas.append((etiqueta[:2], etiqueta, dev))
        elif _FILTRO.match(etiqueta):
            filtros.append((etiqueta, dev))
    return {"cabecera": cabecera, "filtros": filtros, "filas": filas}

def consultar(anio, mes, depto, reintentos=4):
    """POST con reintentos y backoff exponencial. mes='' -> vista anual."""
    params = {"0": "", "24": GENERICA, "23": mes, "1": NIVEL_GOBIERNO, "37": "",
              "5": depto, "21": depto, "8": "", "7": "", "y": anio}
    params.update(PARAMS_EXTRA)
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.post(URL_BASE, params=params, timeout=30)
            if r.status_code == 200:
                p = parsear(r.content)
                if p is not None:
                    return p, None
                ultimo = "SIN_TABLA"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def validar(p, depto):
    """Verifica que la página corresponde a lo pedido. Devuelve (errores, avisos)."""
    errores, avisos = [], []
    etiquetas = [e for e, _ in p["filtros"]]
    if "funci" not in p["cabecera"].lower():
        errores.append(f"cabecera='{p['cabecera']}' (se esperaba 'Función')")
    if not any(e.lower().startswith("nivel de gobierno") and "regional" in e.lower() for e in etiquetas):
        errores.append("la página no está filtrada por GOBIERNOS REGIONALES")
    if not any(e.lower().startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    if not p["filas"]:
        errores.append("sin filas de función")
    if not errores and p["filtros"]:
        dev_dep = p["filtros"][-1][1]                    # última fila de filtros = Departamento
        suma = sum(v for _, _, v in p["filas"])
        tol = max(1, len(p["filas"]))                    # redondeo máximo: S/ 1 por fila
        if abs(suma - dev_dep) > tol:
            avisos.append(f"CUADRE: suma funciones {suma:,.0f} vs fila departamento {dev_dep:,.0f}")
    return errores, avisos

def extraer(anio, mes, depto):
    """-> (registros, error, aviso) para una región y un mes."""
    p, err = consultar(anio, mes, depto)
    if err:
        return [], err, None
    errores, avisos = validar(p, depto)
    if errores:
        return [], "VALIDACIÓN: " + "; ".join(errores), None
    nombre_dep = deptos_map[depto].split(": ", 1)[1]
    regs = [{
        "AÑO": anio, "MES": meses_map[mes], "NIVEL_GOBIERNO": NIVEL_ETIQUETA,
        "DEPARTAMENTO": deptos_map[depto], "ENTIDAD": f"GOBIERNO REGIONAL DE {nombre_dep}",
        "FUNCION": nombre, "DEVENGADO": dev, "COD_DEPTO": depto, "COD_FUNCION": cod,
    } for cod, nombre, dev in p["filas"]]
    return regs, None, ("; ".join(avisos) if avisos else None)

# ── 4. CONTROL ANUAL (año x departamento x función x devengado) ──────────────
def control_anual(df_total, anio, mes_ultimo):
    """Compara lo raspado contra la vista anual de Consulta Amigable (sin filtro de mes).
    Detecta además si el Devengado de cada mes es MENSUAL (la suma de meses cuadra)
    o ACUMULADO (el último mes cuadra)."""
    filas = []
    for d in departamentos:
        p, err = consultar(anio, "", d["codigo"])
        errores, _ = validar(p, d["codigo"]) if p else ([err], [])
        if errores:
            print(f"   ❌ Referencia anual {d['departamento']}: {errores}")
            continue
        ref = {cod: dev for cod, _, dev in p["filas"]}
        sub = df_total[df_total["COD_DEPTO"] == d["codigo"]]
        suma = sub.groupby("COD_FUNCION")["DEVENGADO"].sum()
        ult = sub[sub["MES"] == meses_map[mes_ultimo]].groupby("COD_FUNCION")["DEVENGADO"].sum()
        for cod in sorted(set(ref) | set(suma.index)):
            filas.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"], "COD_FUNCION": cod,
                          "DEV_REF_ANUAL": ref.get(cod, 0.0),
                          "DEV_SUMA_MESES": float(suma.get(cod, 0.0)),
                          "DEV_ULTIMO_MES": float(ult.get(cod, 0.0))})
    ctl = pd.DataFrame(filas)
    if ctl.empty:
        return ctl, "SIN REFERENCIA"
    tol = max(5, len(meses))
    ctl["DIF_SUMA"] = ctl["DEV_SUMA_MESES"] - ctl["DEV_REF_ANUAL"]
    ctl["DIF_ULTIMO"] = ctl["DEV_ULTIMO_MES"] - ctl["DEV_REF_ANUAL"]
    ok_suma = ctl["DIF_SUMA"].abs().le(tol).all()
    ok_ult = ctl["DIF_ULTIMO"].abs().le(tol).all()
    if ok_suma:
        semantica, col = "MENSUAL (la suma de los meses cuadra con el anual)", "DIF_SUMA"
    elif ok_ult:
        semantica, col = "ACUMULADO (el último mes cuadra con el anual: NO sumar meses)", "DIF_ULTIMO"
    else:
        semantica, col = "NO CUADRA (revisar)", "DIF_SUMA"
    ctl["ESTADO"] = np.where(ctl[col].abs().le(tol), "OK", "REVISAR")
    ctl["TOLERANCIA_S/"] = tol
    ctl["FECHA_EXTRACCION"] = date.today().isoformat()
    return ctl, semantica

# ── 5. BUCLE MES A MES (regiones en secuencia: Loreto -> San Martín -> Ucayali) ─
registros_todos, errores_totales, avisos_totales = [], [], []
archivos_generados, resumen_meses = [], []

print(f"Iniciando extracción {AÑO}: {len(meses)} mes(es) x {len(departamentos)} región(es) "
      f"= {len(meses) * len(departamentos)} consultas.\n")

for mes in tqdm(meses, desc="Meses"):
    etiqueta = meses_map[mes]
    regs_mes, errs_mes = [], []
    for d in departamentos:
        regs, err, aviso = extraer(AÑO, mes, d["codigo"])
        regs_mes.extend(regs)
        if err:
            errs_mes.append({"AÑO": AÑO, "MES": etiqueta, "DEPARTAMENTO": d["departamento"], "ERROR": err})
        if aviso:
            avisos_totales.append({"AÑO": AÑO, "MES": etiqueta, "DEPARTAMENTO": d["departamento"], "AVISO": aviso})
        time.sleep(random.uniform(*PAUSA))

    errores_totales.extend(errs_mes)
    registros_todos.extend(regs_mes)
    df_mes = pd.DataFrame(regs_mes)
    if df_mes.empty:
        estado = "SIN DATOS"
    else:
        archivo = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, [mes]))
        df_mes[COLUMNAS_SALIDA].to_csv(archivo, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo, DRIVE_FOLDER_ID)
        estado = "OK" if not errs_mes else "INCOMPLETO (con errores)"
    resumen_meses.append({"MES": etiqueta, "FILAS": len(df_mes), "ERRORES": len(errs_mes), "ESTADO": estado})

print("═" * 70)
print(f"RESUMEN {AÑO}")
display(pd.DataFrame(resumen_meses))
if errores_totales:
    print("⚠️ Consultas con error:"); display(pd.DataFrame(errores_totales))
if avisos_totales:
    print("⚠️ Avisos de cuadre interno (suma de funciones ≠ fila del departamento):")
    display(pd.DataFrame(avisos_totales))
if not errores_totales and not avisos_totales:
    print("✅ Sin errores y con cuadre interno en todas las consultas.")

# ── 6. CONTROL ANUAL + CIERRE (guardado, Drive, descarga) ────────────────────
ctl_anual = pd.DataFrame()
df_total = pd.DataFrame(registros_todos)
cubre_anio = (MES_INICIO == 1) and (AÑO < hoy.year or INCLUIR_MES_EN_CURSO)

if CONTROL_ANUAL and df_total.empty:
    print("\nControl anual omitido: no hay datos raspados.")
elif CONTROL_ANUAL and not cubre_anio:
    print("\nℹ️ Control anual omitido: el bucle no cubre el año completo "
          "(MES_INICIO>1 o año en curso con meses cerrados). La vista anual del MEF incluye "
          "el mes en curso; usa INCLUIR_MES_EN_CURSO=True para cuadrarla el mismo día.")
elif CONTROL_ANUAL:
    print("\n🔍 Control anual contra Consulta Amigable (3 consultas)…")
    ctl_anual, semantica = control_anual(df_total, AÑO, meses[-1])
    print(f"   Semántica detectada del Devengado por mes: {semantica}")
    if not ctl_anual.empty:
        display(ctl_anual["ESTADO"].value_counts().to_frame("funciones"))
        display(ctl_anual[ctl_anual["ESTADO"] == "REVISAR"])
        display(ctl_anual.groupby("DEPARTAMENTO")[["DEV_REF_ANUAL", "DEV_SUMA_MESES", "DEV_ULTIMO_MES"]].sum())
        archivo_ctl = os.path.join(ruta_guardado, f"CONTROL GobRegional {AÑO}.csv")
        ctl_anual.to_csv(archivo_ctl, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo_ctl)
        if SUBIR_A_DRIVE:
            subir_a_drive(drive, archivo_ctl, DRIVE_FOLDER_ID)

if SUBIR_A_DRIVE and (errores_totales or avisos_totales):
    log = pd.concat([pd.DataFrame(errores_totales), pd.DataFrame(avisos_totales)], ignore_index=True)
    archivo_log = os.path.join(ruta_guardado, f"LOG GobRegional {AÑO} - {etiqueta_meses(meses)}.csv")
    log.to_csv(archivo_log, index=False, encoding="utf-8-sig")
    subir_a_drive(drive, archivo_log, DRIVE_FOLDER_ID)

if SUBIR_A_DRIVE:
    print(f"\n☁️ DRIVE: {len(drive_subidos)} subido(s), {len(drive_fallidos)} fallido(s)")
    print(f"   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")

if not archivos_generados:
    print("\n⚠️ No hay archivos para descargar.")
elif DESCARGA_FINAL == "zip":
    from google.colab import files
    ruta_zip = os.path.join(ruta_guardado, nombre_archivo_salida(AÑO, meses).replace(".csv", ".zip"))
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for a in archivos_generados:
            z.write(a, arcname=os.path.basename(a))
    files.download(ruta_zip)
elif DESCARGA_FINAL == "individual":
    from google.colab import files
    for a in archivos_generados:
        files.download(a)
else:
    print(f"\n💾 Archivos guardados en {ruta_guardado}")

Año 2025 -> meses a procesar: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12  |  regiones: 16: LORETO, 22: SAN MARTÍN, 25: UCAYALI


☁️  Drive conectado -> carpeta: 'data'

Iniciando extracción 2025: 12 mes(es) x 3 región(es) = 36 consultas.



Meses:   0%|          | 0/12 [00:00<?, ?it/s]

══════════════════════════════════════════════════════════════════════
RESUMEN 2025


,MES,FILAS,ERRORES,ESTADO
0,1: Enero,0,3,SIN DATOS
1,2: Febrero,0,3,SIN DATOS
2,3: Marzo,0,3,SIN DATOS
3,4: Abril,0,3,SIN DATOS
4,5: Mayo,0,3,SIN DATOS
5,6: Junio,0,3,SIN DATOS
6,7: Julio,0,3,SIN DATOS
7,8: Agosto,0,3,SIN DATOS
8,9: Setiembre,0,3,SIN DATOS
9,10: Octubre,0,3,SIN DATOS


⚠️ Consultas con error:


,AÑO,MES,DEPARTAMENTO,ERROR
0,2025,1: Enero,16: LORETO,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
1,2025,1: Enero,22: SAN MARTÍN,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
2,2025,1: Enero,25: UCAYALI,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
3,2025,2: Febrero,16: LORETO,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
4,2025,2: Febrero,22: SAN MARTÍN,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
5,2025,2: Febrero,25: UCAYALI,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
6,2025,3: Marzo,16: LORETO,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
7,2025,3: Marzo,22: SAN MARTÍN,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
8,2025,3: Marzo,25: UCAYALI,VALIDACIÓN: cabecera='' (se esperaba 'Función'...
9,2025,4: Abril,16: LORETO,VALIDACIÓN: cabecera='' (se esperaba 'Función'...



Control anual omitido: no hay datos raspados.
   ☁️ Drive (creado): LOG GobRegional 2025 - Meses 01-12.csv

☁️ DRIVE: 1 subido(s), 0 fallido(s)
   https://drive.google.com/drive/folders/18Ohcjmg6FojhNwbCHUHX3MEOxhokZA9b

⚠️ No hay archivos para descargar.


In [3]:
# ═══ CELDA DE DIAGNÓSTICO (independiente: no necesita ejecutar el Bloque 2) ═══
import re, requests
from bs4 import BeautifulSoup

URL = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
BASE = {"0": "", "24": "6-2-6", "23": "1", "1": "R", "37": "", "5": "16", "21": "16",
        "8": "", "7": "", "y": 2025}
# None = quitar el parámetro; "" = enviarlo vacío
VARIANTES = {
    "A · actual (5 y 21 = 16)":        {},
    "B · solo '5' = 16":               {"21": ""},
    "C · solo '21' = 16":              {"5": ""},
    "D · sin parámetro '37'":          {"37": None},
    "E · sin '37' y solo '21'":        {"37": None, "5": ""},
    "F · '37' = R":                    {"37": "R"},
}
FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|funci[oó]n|"
                    r"mes\b|sector|pliego|fuente|rubro|categor|producto|divisi[oó]n)", re.I)

def num(t):
    t = (t or "").replace(",", "").replace("\xa0", "").strip()
    try: return float(t)
    except ValueError: return None

def radiografia(html):
    s = BeautifulSoup(html, "html.parser")
    tabla = s.find("table", class_="Data")
    info = {"tabla_Data": tabla is not None, "n_th": len(s.find_all("th")),
            "aparece_'Función'": "función" in s.get_text().lower(), "breadcrumb": [], "filas": []}
    for tr in s.find_all("tr"):
        if tr.find("table"):
            continue
        celdas = tr.find_all("td")
        if len(celdas) < 8:
            continue
        txt = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:2]]
        et = next((t for t in txt if t), "")
        dev = num(celdas[-3].get_text(strip=True))
        if dev is None:
            continue
        (info["breadcrumb"] if FILTRO.match(et) else info["filas"]).append((et[:60], dev, len(celdas)))
    return info

sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                                     "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"})
ganadoras = []
for nombre, cambios in VARIANTES.items():
    params = {**BASE, **cambios}
    params = {k: v for k, v in params.items() if v is not None}
    print("═" * 78); print(nombre)
    try:
        r = sesion.post(URL, params=params, timeout=40)
    except Exception as e:
        print("   ❌ error de red:", e); continue
    info = radiografia(r.content)
    print(f"   HTTP {r.status_code} | bytes={len(r.content):,} | tabla.Data={info['tabla_Data']} | "
          f"<th>={info['n_th']} | 'Función' en página={info['aparece_\'Función\'']}")
    print("   Filas de filtro (breadcrumb):")
    for et, dev, n in info["breadcrumb"]:
        print(f"      [{n} celdas] {et}  ->  {dev:,.0f}")
    print(f"   Filas de detalle: {len(info['filas'])}  (primeras 6)")
    for et, dev, n in info["filas"][:6]:
        print(f"      [{n} celdas] {et}  ->  {dev:,.0f}")
    bc = " | ".join(e for e, _, _ in info["breadcrumb"]).lower()
    if "departamento" in bc and "16:" in bc and "regional" in bc and len(info["filas"]) >= 5:
        print("   ✅ ESTA VARIANTE FUNCIONA"); ganadoras.append(nombre)
    with open(f"/content/respuesta_{nombre[0]}.html", "wb") as f:
        f.write(r.content)

print("═" * 78)
print("VARIANTES QUE FUNCIONAN:", ganadoras if ganadoras else "NINGUNA -> sube respuesta_A.html y lee las instrucciones")
try:
    from google.colab import files
    files.download("/content/respuesta_A.html")
except Exception:
    pass

══════════════════════════════════════════════════════════════════════════════
A · actual (5 y 21 = 16)
   HTTP 200 | bytes=52,724 | tabla.Data=True | <th>=0 | 'Función' en página=True
   Filas de filtro (breadcrumb):
      [10 celdas] TOTAL  ->  253,675,294,538
      [10 celdas] Genérica 6-26: ADQUISICION DE ACTIVOS NO FINANCIEROS  ->  61,623,514,444
      [10 celdas] Mes 1: enero  ->  3,170,422,448
      [10 celdas] Nivel de Gobierno R: GOBIERNOS REGIONALES  ->  630,119,022
   Filas de detalle: 0  (primeras 6)
══════════════════════════════════════════════════════════════════════════════
B · solo '5' = 16
   HTTP 200 | bytes=52,714 | tabla.Data=True | <th>=0 | 'Función' en página=True
   Filas de filtro (breadcrumb):
      [10 celdas] TOTAL  ->  253,675,294,538
      [10 celdas] Genérica 6-26: ADQUISICION DE ACTIVOS NO FINANCIEROS  ->  61,623,514,444
      [10 celdas] Mes 1: enero  ->  3,170,422,448
      [10 celdas] Nivel de Gobierno R: GOBIERNOS REGIONALES  ->  630,119,022
   Filas

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>